In [1]:
# Notebook 02: Lam sach du lieu recipes va interactions

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from pathlib import Path
from collections import Counter

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

# Tim duong dan data/raw de chay duoc ca tu repo root va notebooks
candidate_paths = [
    Path.cwd() / 'data' / 'raw',
    Path.cwd().parent / 'data' / 'raw'
]
raw_dir = next((p for p in candidate_paths if p.exists()), None)
if raw_dir is None:
    raise FileNotFoundError("Khong tim thay thu muc data/raw. Hay kiem tra lai duong dan du lieu.")

df_recipes = pd.read_csv(raw_dir / 'RAW_recipes.csv')
df_interactions = pd.read_csv(raw_dir / 'RAW_interactions.csv')

print(f"Recipes: {df_recipes.shape[0]} dòng x {df_recipes.shape[1]} cột")
print(f"Interactions: {df_interactions.shape[0]} dòng x {df_interactions.shape[1]} cột")

Recipes: 231637 dòng x 12 cột
Interactions: 1132367 dòng x 5 cột


In [3]:
nutrition_cols = [
    "calories",
    "fat",
    "sugar",
    "sodium",
    "protein",
    "saturated_fat",
    "carbs"
]

df_recipes[nutrition_cols] = pd.DataFrame(
    df_recipes['nutrition'].apply(ast.literal_eval).tolist(),
    index=df_recipes.index
)

df_recipes[nutrition_cols].describe()



,calories,fat,sugar,sodium,protein,saturated_fat,carbs
count,231637.00,231637.00,231637.00,231637.00,231637.00,231637.00,231637.00
mean,473.94,36.08,84.30,30.15,34.68,45.59,15.56
std,1189.71,77.80,800.08,131.96,58.47,98.24,81.82
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,174.40,8.00,9.00,5.00,7.00,7.00,4.00
50%,313.40,20.00,25.00,14.00,18.00,23.00,9.00
75%,519.70,41.00,68.00,33.00,51.00,52.00,16.00
max,434360.20,17183.00,362729.00,29338.00,6552.00,10395.00,36098.00


In [4]:
# Xu ly gia tri thieu

df_recipes.isnull().sum()
(df_recipes.isnull().sum() / len(df_recipes)) * 100

df_recipes['description'] = df_recipes['description'].fillna("No description available")


In [5]:
# Lam sach recipes va interactions cho baseline

def parse_list_safe(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return []

report = {
    "recipes_before": len(df_recipes),
    "interactions_before": len(df_interactions),
}

# Recipes
recipes = df_recipes.copy()
recipes = recipes.drop_duplicates(subset=["id"], keep="first").copy()
recipes["name"] = recipes["name"].fillna("").str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
recipes["description"] = recipes["description"].fillna("No description available")
recipes["calories"] = pd.to_numeric(recipes["calories"], errors="coerce")
recipes["minutes"] = pd.to_numeric(recipes["minutes"], errors="coerce")

for col in ["ingredients", "tags", "steps"]:
    recipes[col] = recipes[col].apply(parse_list_safe)

# Tao cac cot text/list da chuan hoa
recipes["ingredients_clean"] = recipes["ingredients"].apply(lambda items: [str(x).strip().lower() for x in items])
recipes["tags_clean"] = recipes["tags"].apply(lambda items: [str(x).strip().lower() for x in items])
recipes["n_ingredients_calc"] = recipes["ingredients_clean"].apply(len)
recipes["ingredients_text"] = recipes["ingredients_clean"].apply(lambda x: " ".join(x))
recipes["tags_text"] = recipes["tags_clean"].apply(lambda x: " ".join(x))

# Interactions
interactions = df_interactions.copy()
interactions = interactions[["user_id", "recipe_id", "date", "rating", "review"]].copy()

interactions["user_id"] = pd.to_numeric(interactions["user_id"], errors="coerce")
interactions["recipe_id"] = pd.to_numeric(interactions["recipe_id"], errors="coerce")
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")
interactions["date"] = pd.to_datetime(interactions["date"], errors="coerce")

interactions = interactions.dropna(subset=["user_id", "recipe_id", "rating"])
interactions["user_id"] = interactions["user_id"].astype("int64")
interactions["recipe_id"] = interactions["recipe_id"].astype("int64")
interactions = interactions[interactions["rating"].between(1, 5, inclusive="both")].copy()
interactions = interactions.drop_duplicates(subset=["user_id", "recipe_id", "date", "rating"], keep="last").copy()

# Dong bo interactions voi recipes
valid_recipe_ids = set(recipes["id"].tolist())
interactions = interactions[interactions["recipe_id"].isin(valid_recipe_ids)].copy()

# Loc sparse
min_user_interactions = 3
min_recipe_interactions = 3

def iterative_sparse_filter(df, min_u=3, min_i=3):
    while True:
        n_before = len(df)
        u = df["user_id"].value_counts()
        df = df[df["user_id"].isin(u[u >= min_u].index)]
        i = df["recipe_id"].value_counts()
        df = df[df["recipe_id"].isin(i[i >= min_i].index)]
        if len(df) == n_before:
            break
    return df.reset_index(drop=True)

interactions = iterative_sparse_filter(interactions)

# Loc recipes theo rang buoc hop ly
recipes = recipes[recipes["minutes"].between(1, 1440)].copy()
recipes = recipes[recipes["calories"].between(10, 5000)].copy()

# Chi giu recipes co xuat hien trong interactions
recipes = recipes[recipes["id"].isin(interactions["recipe_id"].unique())].copy()
valid_recipe_ids = set(recipes["id"])
interactions = interactions[interactions["recipe_id"].isin(valid_recipe_ids)].copy()

interactions["review"] = interactions["review"].fillna("").astype(str).str.strip()

report.update({
    "recipes_after": len(recipes),
    "interactions_after": len(interactions),
    "n_unique_users_after": interactions["user_id"].nunique(),
    "n_unique_recipes_after": interactions["recipe_id"].nunique(),
    "min_user_interactions": min_user_interactions,
    "min_recipe_interactions": min_recipe_interactions,
})

print("Cleaning report:")
for k, v in report.items():
    print(f"- {k}: {v}")

display(recipes.head(2))
display(interactions.head(2))

Cleaning report:
- recipes_before: 231637
- interactions_before: 1132367
- recipes_after: 76608
- interactions_after: 693880
- n_unique_users_after: 30836
- n_unique_recipes_after: 76608
- min_user_interactions: 3
- min_recipe_interactions: 3


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,...,sugar,sodium,protein,saturated_fat,carbs,ingredients_clean,tags_clean,n_ingredients_calc,ingredients_text,tags_text
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, main-ingredie...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depending on siz...",autumn is my favorite time of year to cook! this recipe ...,...,13.00,0.00,2.00,0.00,4.00,"[winter squash, mexican seasoning, mixed spice, honey, b...","[60-minutes-or-less, time-to-make, course, main-ingredie...",7,winter squash mexican seasoning mixed spice honey butter...,60-minutes-or-less time-to-make course main-ingredient c...
9,beat this banana bread,75452,70,15892,2003-11-04,"[weeknight, time-to-make, course, main-ingredient, prepa...","[2669.3, 160.0, 976.0, 107.0, 62.0, 310.0, 138.0]",12,"[preheat oven to 350 degrees, butter two 9x5"" loaf pans,...",from ann hodgman's,...,976.00,107.00,62.00,310.00,138.00,"[sugar, unsalted butter, bananas, eggs, fresh lemon juic...","[weeknight, time-to-make, course, main-ingredient, prepa...",9,sugar unsalted butter bananas eggs fresh lemon juice ora...,weeknight time-to-make course main-ingredient preparatio...


,user_id,recipe_id,date,rating,review
0,76535,134728,2005-09-02,4,Very good!
1,353911,134728,2006-09-26,5,Absolutely AWESOME! I was speechless when I tried them. ...


In [6]:
# Tach recipes du tuong tac cho CF va recipes it tuong tac (cold)
all_recipe_ids_after_sync = set(interactions["recipe_id"].unique())
interactions_filtered = iterative_sparse_filter(interactions)
active_recipe_ids = set(interactions_filtered["recipe_id"].unique())

cold_recipes = recipes[
    recipes["id"].isin(all_recipe_ids_after_sync - active_recipe_ids)
].copy()

print("Active recipes:", len(active_recipe_ids))
print("Cold recipes:", len(cold_recipes))

Active recipes: 76545
Cold recipes: 63


In [7]:
def final_validation(recipes, interactions):
    errors = []

    core_r = ["id", "name", "ingredients_clean", "tags_clean", "calories", "ingredients_text", "tags_text"]
    for col in core_r:
        n = recipes[col].isnull().sum()
        if n > 0:
            errors.append(f"recipes[{col}] null: {n}")

    core_i = ["user_id", "recipe_id", "rating"]
    for col in core_i:
        n = interactions[col].isnull().sum()
        if n > 0:
            errors.append(f"interactions[{col}] null: {n}")

    if (interactions["rating"] == 0).any():
        errors.append("rating = 0")

    orphan = set(interactions["recipe_id"]) - set(recipes["id"])
    if orphan:
        errors.append(f"orphan recipe_id: {len(orphan)}")

    bad_cal = (~recipes["calories"].between(10, 5000)).sum()
    if bad_cal > 0:
        errors.append(f"calories out of range: {bad_cal}")

    if errors:
        print("Validation failed")
        for e in errors:
            print("-", e)
        return False

    print("Validation passed")
    print("recipes:", len(recipes))
    print("interactions:", len(interactions))
    print("users:", interactions["user_id"].nunique())
    print("avg_rating:", round(interactions["rating"].mean(), 2))
    return True


validation_passed = final_validation(recipes, interactions)

Validation passed
recipes: 76608
interactions: 693880
users: 30836
avg_rating: 4.71


In [8]:
candidate_processed = [
    Path.cwd() / "data" / "processed",
    Path.cwd().parent / "data" / "processed",
]

processed_dir = candidate_processed[0]
if not processed_dir.parent.exists():
    processed_dir = candidate_processed[1]

if not validation_passed:
    raise RuntimeError("Validation failed, stop saving.")

processed_dir.mkdir(parents=True, exist_ok=True)

recipes_out = recipes.copy()
interactions_out = interactions.copy()

for col in ["ingredients_clean", "tags_clean", "ingredients", "tags", "steps"]:
    recipes_out[col] = recipes_out[col].apply(str)

recipes_out.to_csv(processed_dir / "recipes_clean.csv", index=False)
interactions_out.to_csv(processed_dir / "interactions_clean.csv", index=False)

print("Saved:", processed_dir / "recipes_clean.csv")
print("Saved:", processed_dir / "interactions_clean.csv")

Saved: d:\GITHUB\Food-RecommendationSystem\data\processed\recipes_clean.csv
Saved: d:\GITHUB\Food-RecommendationSystem\data\processed\interactions_clean.csv
